## Portability and scientific-review status

This notebook has been cleared of saved execution state and personal paths. Its original calculations are retained for portability review. Before using its metrics for scientific acceptance, review finite-pair filtering, categorical nearest-neighbor region assignment, and the target-diagram variability sign.


In [ ]:
# Script of postprocessing plots for validation tables

#Example of script if you want to play with your validation tables and see what 
#the validation looks like, per regions (bias, RMSD, correlation, Taylor diagrams,...).  
#This script will be mainly used by WP4.  

#- Input file format: VALID_{var}_ {year}_{model}.parquet  

#_Authors: Capet A._

#Modified by Kenny L. with help from Codex

In [ ]:
from pathlib import Path
import warnings

import xarray as xr
import matplotlib.pyplot as plt

from matplotlib.colors import LogNorm
from matplotlib.ticker import LogFormatter

import numpy as np
import pandas as pd

from cmocean import cm
from matplotlib.colors import ListedColormap

from cartopy import crs as ccrs
import cartopy.feature as cf

import datetime


def managed_output_path(path, save_enabled, overwrite=False):
    """Return a writable path, or warn and return None without overwriting."""
    path = Path(path)

    if path.exists() and not overwrite:
        warnings.warn(
            f"File not saved: file already exists: {path}",
            stacklevel=2,
        )
        return None

    if not save_enabled:
        warnings.warn(
            f"File not saved: output writing is disabled: {path}",
            stacklevel=2,
        )
        return None

    path.parent.mkdir(parents=True, exist_ok=True)
    return path

In [ ]:
# ============================================================================
# MANAGER CELL 1: original regional-validation workflow
# Change settings here; downstream original-workflow cells use these values.
# ============================================================================

setup_root = Path(
    "/absolute/path/to/CodeBlueWP4-KLL_update/validations/Test"
)
datadir = setup_root / "output"
regionfile = (
    setup_root / "input_data" / "regions"
    / "Bathymetry_NoS_with_regions_AC.nc"
)

var = "talk" #["oxy", "nox", "nh4", "po4", "sio", "chl", "temp", "sal", "talk"]
year = 2014
model = "test"
refresh_region_ids = True
maxdepth = 20

# False is safest for reviewing existing results. Set True only when creating
# deliberately new original-workflow products. Existing files remain protected
# unless overwrite_original_outputs is also changed explicitly.
save_original_outputs = True
overwrite_original_outputs = True

## Load regions

In [ ]:
warnings.filterwarnings(
    "ignore",
    message=r'facecolor will have no effect as it has been defined as "never"\.',
    category=UserWarning,
    module=r"cartopy\.mpl\.feature_artist",
)

xreg = xr.load_dataset(regionfile)
nreg = int(xreg.region_id.max().values + 1)
print(f"Regions: {nreg}")

regcmap = ListedColormap(plt.cm.Spectral(np.linspace(0, 1, nreg)))

fig = plt.figure(figsize=(15, 8))
ax = fig.add_subplot(111, projection=ccrs.PlateCarree())
xreg.region_id.plot(cmap=regcmap)

ax.gridlines()
ax.coastlines(color="k")
ax.add_feature(cf.BORDERS, color="darkgrey")
ax.add_feature(cf.RIVERS, color="darkblue")
plt.show()

In [ ]:
validation_path = datadir / f"VALID_{var}_{year}_{model}.parquet"
if not validation_path.is_file():
    raise FileNotFoundError(f"Missing validation table: {validation_path}")

dflt = pd.read_parquet(validation_path)

if refresh_region_ids:
    lons = xr.DataArray(dflt["lon"], dims="points")
    lats = xr.DataArray(dflt["lat"], dims="points")
    times = xr.DataArray(dflt["datetime"], dims="points")
    depths = xr.DataArray(dflt["depth"], dims="points")

    # This preserves the original notebook behavior. The separate reviewed
    # COMP4 workflow uses nearest-neighbour assignment for categorical IDs.
    regs = xreg["region_id"].interp({"lon": lons, "lat": lats})
    dflt["reg"] = np.round(regs)

In [ ]:
dflt = dflt.dropna()

In [ ]:
dflt.groupby('reg').count()['lon']

In [ ]:
regdatacount = np.zeros(nreg)

for r in range(nreg):
    regdatacount[r] = dflt[dflt['reg'] == r].count()['lon']

plt.bar(np.arange(nreg), regdatacount)
plt.xlabel('Regions')
plt.ylabel('Data counts')


In [ ]:
plt.scatter(regs['lon'],regs['lat'],50, regs, cmap=regcmap)

In [ ]:
fig = plt.figure(figsize = (15,6))

# Left panel wtih regions 
ax= fig.add_subplot(1,2,1)

dflt.plot.scatter(x=var,y='mod', c = 'reg', ax=ax, cmap=regcmap)
ax.grid()

xmin, xmax = min(dflt[var].min(), dflt['mod'].min()), max(dflt[var].max(), dflt['mod'].max()) 
ax.set_xlim((xmin,xmax))
ax.set_ylim((xmin,xmax))
ax.set_ylabel('Model Values')
ax.set_xlabel('Observed Values')

ax.set_title(var)
ax. plot([xmin,xmax], [xmin,xmax], 'r--')

# Depth
ax= fig.add_subplot(1,2,2)

dflt.plot.scatter(x=var,y='mod', c = 'depth', ax=ax, cmap=cm.algae)
ax.grid()

xmin, xmax = min(dflt[var].min(), dflt['mod'].min()), max(dflt[var].max(), dflt['mod'].max()) 
ax.set_xlim((xmin,xmax))
ax.set_ylim((xmin,xmax))
ax.set_ylabel('Model Values')
ax.set_xlabel('Observed Values')

ax.set_title(var)
ax. plot([xmin,xmax], [xmin,xmax], 'r--')


plt.show()

## Validation Metric


For each region, based on a DOY timeseries, recomposed from MULTIYEAR regional spatial average for model and (availale) obs  : 
* BIAS
* RMSD
* CORR

-> Taylor / Target for regional CHL. 



In [ ]:
stats_list = []

dflt["diff"] = dflt["mod"] - dflt[var]

for reg in range(nreg):
    dfloc = dflt[dflt["reg"] == reg]

    mod_mean = dfloc["mod"].mean()
    mod_std = dfloc["mod"].std()
    obs_mean = dfloc[var].mean()
    obs_std = dfloc[var].std()
    bias = mod_mean - obs_mean
    perc_bias = np.nan if obs_mean == 0 else (bias / obs_mean) * 100
    corr = (
        np.nan
        if len(dfloc) < 2 or dfloc["mod"].nunique() < 2 or dfloc[var].nunique() < 2
        else np.corrcoef(dfloc["mod"], dfloc[var], rowvar=False)[0, 1]
    )
    denominator = ((dfloc[var] - obs_mean) ** 2).sum()
    nse = np.nan if denominator == 0 else 1 - (dfloc["diff"] ** 2).sum() / denominator
    rmsd = np.sqrt((dfloc["diff"] ** 2).mean())
    crmsd = np.sqrt(
        (((dfloc[var] - obs_mean) - (dfloc["mod"] - mod_mean)) ** 2).mean()
    )

    stats_list.append({
        "region": reg,
        "model_mean": mod_mean,
        "model_std": mod_std,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "bias": bias,
        "perc_bias": perc_bias,
        "correlation": corr,
        "NSE": nse,
        "RMSD": rmsd,
        "cRMSD": crmsd,
        "N": regdatacount[reg],
    })

stats_df = pd.DataFrame(stats_list)

stats_output = managed_output_path(
    datadir / f"validation_metrics_{year}_{var}.csv",
    save_enabled=save_original_outputs,
    overwrite=overwrite_original_outputs,
)
if stats_output is not None:
    stats_df.to_csv(stats_output, index=False)
    print(f"Saved: {stats_output}")

stats_df

In [ ]:
stats_df

In [ ]:
# Initialize an empty dictionary to store metric data
metric_data = {}

# Loop through each metric and store it in the dictionary
for metric in ['bias', 'perc_bias', 'correlation', 'NSE', 'RMSD','cRMSD']:
    # Create a 2D array for each region
    metric_values = np.full_like(xreg['region_id'], np.nan, dtype=np.float32)
    
    # Fill metric values for each region
    for reg in range(nreg):
        # Get the metric value for the current region
        metric_values[xreg['region_id'] == reg] = stats_df.loc[stats_df['region'] == reg, metric]#.values
    
    # Add to the dictionary with the metric name
    metric_data[metric] = (['lat', 'lon'], metric_values)

# Create the xarray Dataset
metrics_ds = xr.Dataset(
    metric_data,
    coords={'lat': xreg.lat, 'lon': xreg.lon},
    attrs={'description': 'Validation metrics for satellite and model comparisons'}
)


In [ ]:
# metric_properties = {
#     'bias': {'cmap': 'coolwarm', 'vmin': -5, 'vmax': 5},
#     'perc_bias': {'cmap': 'coolwarm', 'vmin': -100, 'vmax': 100},
#     'correlation': {'cmap': 'RdBu_r', 'vmin': -1, 'vmax': 1},
#     'NSE': {'cmap': 'YlGnBu', 'vmin': -50, 'vmax': 1},
#     'RMSD': {'cmap': 'viridis', 'vmin': 0, 'vmax': 5}
# }

metric_properties = {
    'bias': {'cmap': 'coolwarm', 'vmin': -np.max(np.abs(metrics_ds['bias'])).values, 'vmax': np.max(np.abs(metrics_ds['bias'])).values},
    'perc_bias': {'cmap': 'coolwarm', 'vmin': -np.max(np.abs(metrics_ds['perc_bias'])).values, 'vmax': np.max(np.abs(metrics_ds['perc_bias'])).values},
    'correlation': {'cmap': 'RdBu_r', 'vmin': -1, 'vmax': 1},
    'NSE': {'cmap': 'RdBu_r', 'vmin': -1, 'vmax': 1},
    'RMSD': {'cmap': 'viridis', 'vmin': 0, 'vmax': np.max(np.abs(metrics_ds['RMSD'])).values}
}

from matplotlib.patches import Circle

# Create a figure with 5 subplots (one for each metric)
fig, axs = plt.subplots(2, 3, figsize=(13, 8))
axs = axs.flatten()

# Loop through each metric and plot it
for i, metric in enumerate(['bias', 'perc_bias', 'correlation', 'NSE', 'RMSD']):
    # Retrieve colormap and limits from the dictionary
    cmap = metric_properties[metric]['cmap']
    vmin = metric_properties[metric]['vmin']
    vmax = metric_properties[metric]['vmax']
    
    # Plot each metric (use xarray's plot method with dynamic cmap and limits)
    metrics_ds[metric].plot(ax=axs[i], cmap=cmap, vmin=vmin, vmax=vmax, cbar_kwargs={'label': metric})
    
    # Set title for each subplot
    axs[i].set_title(f'{metric.capitalize()}')

ax=axs[5]

ax.scatter(stats_df['cRMSD']/stats_df['obs_std']*np.sign(1-stats_df['model_std']/stats_df['obs_std'] ),
            stats_df['bias']/stats_df['obs_std'], stats_df['N'], c = stats_df['region'], cmap=regcmap, alpha=0.7)

for reg in range(nreg):
    dloc = stats_df[stats_df.region==reg]
    # ax.text(dloc['cRMSD'].item()/dloc['obs_std'].item()*np.sign(1-dloc['model_std'].item()/dloc['obs_std'].item()),
    #     dloc['bias'].item()/dloc['obs_std'].item(),
    #     '%s'%dloc['region'].item(), ha = 'center', va='center' )

# Centering the axes and adding lines and circle
scale=5
ax.set_xlim([-scale,scale])  # Adjust the limits to fit your data
ax.set_ylim([-scale,scale])

# Adding vertical and horizontal lines at x=0, y=0
ax.axhline(0, color='black',linewidth=1)
ax.axvline(0, color='black',linewidth=1)

# Adding a circle with radius 1 centered at (0, 0)
circle = Circle((0, 0), 1, color='r', fill=False, linestyle='--')
ax.add_patch(circle)

# Set labels
ax.set_xlabel('Normalized RMSD')
ax.set_ylabel('Normalized Bias')
      
# Adjust layout to make room for colorbars
fig.tight_layout()

# Save only when enabled in Manager Cell 1.
metrics_map_output = managed_output_path(
    datadir / f"Validation_Metrics_Maps_xarray_{var}_{year}.png",
    save_enabled=save_original_outputs,
    overwrite=overwrite_original_outputs,
)
if metrics_map_output is not None:
    fig.savefig(metrics_map_output, dpi=200, bbox_inches="tight")
    print(f"Saved: {metrics_map_output}")
plt.show()

In [ ]:
# Create a figure with 2 rows and 4 columns, with one larger panel for the target diagram
fig, ax = plt.subplots(1,1, figsize=(6, 6))

# Scatter plot for the target diagram
ax.scatter(stats_df['cRMSD']/stats_df['obs_std']*np.sign(1-stats_df['model_std']/stats_df['obs_std'] ), stats_df['bias']/stats_df['obs_std'])

# Add region-specific text labels
#for reg in range(nreg):
#    dloc = stats_df[stats_df.region==reg]
#    ax.text(dloc['cRMSD'].item()/dloc['obs_std'].item()*np.sign(1-dloc['model_std'].item()/dloc['obs_std'].item()),
#        dloc['bias'].item()/dloc['obs_std'].item(),
#        '%s'%dloc['region'].item(), ha = 'center', va='center' )

for reg in range(nreg):
    dloc = stats_df[stats_df.region == reg]

    if dloc.empty or not np.isfinite(dloc["obs_std"].item()) or dloc["obs_std"].item() <= 0:
        continue

    obs_std = dloc["obs_std"].item()

    ax.text(
        dloc["cRMSD"].item() / obs_std
        * np.sign(1 - dloc["model_std"].item() / obs_std),
        dloc["bias"].item() / obs_std,
        "%s" % dloc["region"].item(),
        ha="center",
        va="center",
    )

# Centering the axes and adding lines and circle
ax.set_xlim([-7, 7])  # Adjust the limits to fit your data
ax.set_ylim([-7, 7])

# Adding vertical and horizontal lines at x=0, y=0
ax.axhline(0, color='black', linewidth=1)
ax.axvline(0, color='black', linewidth=1)

# Adding a circle with radius 1 centered at (0, 0)
circle = Circle((0, 0), 1, color='r', fill=False, linestyle='--')
ax.add_patch(circle)

# Set labels for the target diagram
ax.set_xlabel('Normalized RMSD')
ax.set_ylabel('Normalized Bias')

# Adjust layout to make room for colorbars and to avoid overlap
fig.tight_layout()

# Save only when enabled in Manager Cell 1.
target_output = managed_output_path(
    datadir / f"Validation_Metrics_Target_{var}_{year}.png",
    save_enabled=save_original_outputs,
    overwrite=overwrite_original_outputs,
)
if target_output is not None:
    fig.savefig(target_output, dpi=200, bbox_inches="tight")
    print(f"Saved: {target_output}")
plt.show()

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

months = list(range(1, 13))
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

ncol = 1

fig, axs = plt.subplots(int(np.ceil(nreg/ncol)), ncol, figsize=(10, 2*nreg), sharex=True)
axs = axs.flatten()

for reg in range(nreg):
    ax = axs[reg]

    dloc = dflt[(dflt['reg'] == reg) & (dflt['depth'] <= maxdepth)].copy()

    # Extract month
    dloc['month'] = pd.Categorical(
        dloc['datetime'].dt.strftime('%b'),
        categories=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],ordered=True)

    if not dloc.empty:
        # 👉 reshape to long format
        dlong = dloc.melt(
            id_vars='month',
            value_vars=[var, 'mod'],
            var_name='variable',
            value_name='value'
        )

        # 👉 seaborn boxplot
        sns.violinplot(
            data=dlong,
            x='month',
            y='value',
            hue='variable',
            ax=ax,
            palette={var: 'tab:blue', 'mod': 'tab:orange'},
            dodge = True
        )

    else:
        # 👉 still create empty axis with full months
        ax.set_xticks(months)

    # 👉 formatting (applies to both empty & non-empty)
    ax.set_title(f'Region {reg}')
    ax.set_xticks(range(len(month_labels)))
    ax.set_xticklabels(month_labels)

    # minor ticks BETWEEN months
    ax.set_xticks(np.arange(0.5, 12.5, 1), minor=True)

    # grid on minor ticks (vertical separators)
    ax.grid(which='minor', axis='x', linestyle='-', alpha=0.3)

    # optional: keep horizontal grid
    ax.grid(which='major', axis='y', linestyle='--', alpha=0.5)

    # 👉 avoid repeating legends everywhere
    if reg != 0:
        ax.get_legend().remove() if ax.get_legend() else None

ymin = np.min(dflt[var])
ymax = np.max(dflt[var])
for ax in axs:
    ax.set_ylim(ymin, ymax)

# 👉 single legend for the whole figure
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')

plt.tight_layout()
plt.show()

## Tier 1 all-variable depth review

This section is separate from the original regional exploration above. It reads the eight completed 2014 validation Parquets without modifying them and writes new review products to a dedicated directory. Run these cells in order.

Review order: `temp`, `sal`; then `po4`, `sio`, `nh4`; then `chl`; then sparse `nox`; and finally unit-provisional `oxy`. Low-`N` depth bins are diagnostic only and should not support strong conclusions.

In [ ]:
# ============================================================================
# MANAGER CELL 2: Tier 1 vertical/depth-review workflow
# Change settings here; all downstream Tier 1 cells use these values.
# ============================================================================

review_mode = "create_new"  # "load_existing" or "create_new"
overwrite_review_outputs = True
review_dir = setup_root / "review"
diagnostic_regionfile = regionfile

variables = ["oxy", "nox", "nh4", "po4", "sio", "chl", "temp", "sal", "talk"]
year = 2014
model = "test"
display_vertical_figures = True

depth_edges = [0, 10, 20, 50, 100, np.inf]
depth_labels = ["0-10", "10-20", ">20-50", ">50-100", ">100"]

# Units refer to paired validation-table values. Oxygen remains provisional
# because model conversion and observation-unit metadata are not yet reconciled.
variable_units = {
    "oxy": "ml L$^{-1}$ (provisional)",
    "nox": "$\\mu$mol N L$^{-1}$",
    "nh4": "$\\mu$mol N L$^{-1}$",
    "po4": "$\\mu$mol P L$^{-1}$",
    "sio": "$\\mu$mol Si L$^{-1}$",
    "chl": "$\\mu$g Chl L$^{-1}$",
    "temp": "$^{\\circ}$C",
    "sal": "psu",
    "talk": "unknown"
}



allowed_review_modes = {"create_new", "load_existing"}
if review_mode not in allowed_review_modes:
    raise ValueError(
        f"review_mode must be one of {sorted(allowed_review_modes)}; "
        f"received {review_mode!r}"
    )

expected_outputs = [
    review_dir / "tier1_input_checksums.csv",
    review_dir / "tier1_all_variables_depth_metrics.csv",
    review_dir / "tier1_depth_bias_all_variables.png",
    review_dir / "tier1_depth_rmsd_all_variables.png",
    review_dir / "tier1_review_flags.csv",
]
for variable in variables:
    expected_outputs.extend([
        review_dir / variable / f"{variable}_{year}_depth_metrics.csv",
        review_dir / variable / f"{variable}_{year}_finite_pairs.csv",
        review_dir / variable / f"{variable}_{year}_point_level_diagnostics.png",
        review_dir / variable / f"{variable}_{year}_residual_maps.png",
    ])

if review_mode == "create_new":
    review_dir.mkdir(parents=True, exist_ok=True)
    for variable in variables:
        (review_dir / variable).mkdir(exist_ok=True)

    existing_outputs = [path for path in expected_outputs if path.exists()]
    if existing_outputs and not overwrite_review_outputs:
        formatted = "\n".join(f"  {path}" for path in existing_outputs)
        raise FileExistsError(
            "Refusing to overwrite existing Tier 1 review products:\n" + formatted
        )
    if existing_outputs and overwrite_review_outputs:
        print(f"Overwriting {len(existing_outputs)} existing review products")
else:
    if not review_dir.is_dir():
        raise FileNotFoundError(f"Existing review directory not found: {review_dir}")

    missing_outputs = [path for path in expected_outputs if not path.exists()]
    if missing_outputs:
        formatted = "\n".join(f"  {path}" for path in missing_outputs)
        raise FileNotFoundError(
            "The existing Tier 1 review is incomplete. Missing products:\n" + formatted
        )

    print(f"Read-only review mode: {review_dir}")


def new_output(path):
    """Return a path for a Tier 1 derived output"""
    path = Path(path)
    if review_mode != "create_new":
        raise RuntimeError("Writing is disabled in load_existing mode")
    if path.exists() and not overwrite_review_outputs:
        raise FileExistsError(f"Refusing to overwrite existing product: {path}")

    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def save_vertical_figure(fig, path):
    """Save a new Tier 1 figure or warn and continue during review."""
    output = managed_output_path(
        path,
        save_enabled=(review_mode == "create_new"),
        overwrite=overwrite_review_outputs,
    )
    if output is not None:
        fig.savefig(output, dpi=200, bbox_inches="tight")
        print(f"Saved: {output}")
        return True
    return False

In [ ]:
def _finite_pairs(frame, variable):
    required = {variable, 'mod', 'depth'}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise KeyError(f'{variable}: missing required columns {missing}')

    obs = pd.to_numeric(frame[variable], errors='coerce')
    mod_values = pd.to_numeric(frame['mod'], errors='coerce')
    depths = pd.to_numeric(frame['depth'], errors='coerce')
    pair_mask = np.isfinite(obs.to_numpy()) & np.isfinite(mod_values.to_numpy())

    pairs = pd.DataFrame({
        'observation': obs.loc[pair_mask],
        'model': mod_values.loc[pair_mask],
        'depth': depths.loc[pair_mask],
    })
    pairs['residual'] = pairs['model'] - pairs['observation']
    pairs['absolute_residual'] = pairs['residual'].abs()
    finite_depths = pairs['depth'].where(
        np.isfinite(pairs['depth'].to_numpy(dtype=float))
    )
    pairs['depth_bin'] = pd.cut(
        finite_depths,
        bins=depth_edges,
        labels=depth_labels,
        include_lowest=True,
        right=True,
        ordered=True,
    )

    unbinned = pairs['depth_bin'].isna()
    if unbinned.any():
        examples = pairs.loc[unbinned, 'depth'].head(5).tolist()
        raise ValueError(
            f'{variable}: {int(unbinned.sum())} finite observation/model pairs '
            f'fall outside the configured depth bins; example depths={examples}'
        )
    return pairs


def _metric_row(pairs):
    n = len(pairs)
    if n == 0:
        return {
            'N': 0, 'model_mean': np.nan, 'observation_mean': np.nan,
            'bias': np.nan, 'percent_bias': np.nan, 'MAE': np.nan,
            'RMSD': np.nan, 'centred_RMSD': np.nan,
            'correlation': np.nan, 'NSE': np.nan,
        }

    obs = pairs['observation'].to_numpy(dtype=float)
    mod_values = pairs['model'].to_numpy(dtype=float)
    residual = mod_values - obs
    obs_mean = obs.mean()
    model_mean = mod_values.mean()
    bias = residual.mean()
    percent_bias = np.nan if obs_mean == 0 else 100.0 * bias / obs_mean
    mae = np.mean(np.abs(residual))
    rmsd = np.sqrt(np.mean(residual ** 2))
    centred_rmsd = np.sqrt(
        np.mean(((obs - obs_mean) - (mod_values - model_mean)) ** 2)
    )

    constant_series = np.all(obs == obs[0]) or np.all(mod_values == mod_values[0])
    correlation = (
        np.nan if n < 2 or constant_series
        else np.corrcoef(mod_values, obs)[0, 1]
    )
    nse_denominator = np.sum((obs - obs_mean) ** 2)
    nse = (
        np.nan if nse_denominator == 0
        else 1.0 - np.sum(residual ** 2) / nse_denominator
    )
    return {
        'N': n,
        'model_mean': model_mean,
        'observation_mean': obs_mean,
        'bias': bias,
        'percent_bias': percent_bias,
        'MAE': mae,
        'RMSD': rmsd,
        'centred_RMSD': centred_rmsd,
        'correlation': correlation,
        'NSE': nse,
    }


def depth_metrics(frame, variable):
    """Compute robust metrics for fixed observation-depth bins."""
    pairs = _finite_pairs(frame, variable)
    rows = []
    for label in depth_labels:
        subset = pairs.loc[pairs['depth_bin'] == label]
        rows.append({'depth_bin': label, **_metric_row(subset)})
    return pd.DataFrame(rows)

In [ ]:
import hashlib


def sha256sum(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


tier1_data = {}
tier1_pairs = {}
tier1_tables = {}

if review_mode == "create_new":
    audit_rows = []
    checksum_rows = []
    combined_tables = []

    for variable in variables:
        parquet_path = datadir / f"VALID_{variable}_{year}_{model}.parquet"
        if not parquet_path.is_file():
            raise FileNotFoundError(f"Missing Tier 1 validation table: {parquet_path}")

        frame = pd.read_parquet(parquet_path)
        tier1_data[variable] = frame
        pairs = _finite_pairs(frame, variable)
        metrics = depth_metrics(frame, variable)
        tier1_pairs[variable] = pairs
        tier1_tables[variable] = metrics

        obs_numeric = pd.to_numeric(frame[variable], errors="coerce")
        mod_numeric = pd.to_numeric(frame["mod"], errors="coerce")
        depth_numeric = pd.to_numeric(frame["depth"], errors="coerce")
        finite_obs = np.isfinite(obs_numeric.to_numpy())
        finite_mod = np.isfinite(mod_numeric.to_numpy())
        finite_pair = finite_obs & finite_mod
        finite_depth = np.isfinite(depth_numeric.to_numpy())
        in_depth_range = finite_depth & (depth_numeric.to_numpy() >= depth_edges[0])
        unbinned_count = int(np.sum(finite_pair & ~in_depth_range))

        binned_total = int(metrics["N"].sum())
        usable_total = len(pairs)
        if binned_total != usable_total:
            raise RuntimeError(
                f"{variable}: depth-bin N sum ({binned_total}) does not match "
                f"finite-pair count ({usable_total})"
            )

        metrics.to_csv(
            new_output(review_dir / variable / f"{variable}_{year}_depth_metrics.csv"),
            index=False,
        )
        combined_tables.append(metrics.assign(variable=variable))

        pair_export = frame.loc[finite_pair].copy()
        pair_export["residual"] = mod_numeric.loc[finite_pair] - obs_numeric.loc[finite_pair]
        pair_export["absolute_residual"] = pair_export["residual"].abs()
        pair_export["depth_bin"] = pd.cut(
            depth_numeric.loc[finite_pair],
            depth_edges,
            labels=depth_labels,
            include_lowest=True,
            right=True,
            ordered=True,
        )
        pair_export.to_csv(
            new_output(review_dir / variable / f"{variable}_{year}_finite_pairs.csv"),
            index=False,
        )

        audit_rows.append({
            "variable": variable,
            "input_rows": len(frame),
            "finite_observations": int(finite_obs.sum()),
            "finite_model_values": int(finite_mod.sum()),
            "finite_pairs": int(finite_pair.sum()),
            "missing_or_nonfinite_observation": int((~finite_obs).sum()),
            "missing_or_nonfinite_model": int((~finite_mod).sum()),
            "missing_or_nonfinite_pair": int((~finite_pair).sum()),
            "unbinned_finite_pairs": unbinned_count,
        })
        checksum_rows.append({
            "variable": variable,
            "input_file": str(parquet_path),
            "bytes": parquet_path.stat().st_size,
            "sha256": sha256sum(parquet_path),
        })
        print(
            f"{variable}: usable finite pairs={usable_total:,}; "
            f"depth-bin N={binned_total:,}"
        )

    combined_metrics = pd.concat(combined_tables, ignore_index=True)
    combined_metrics = combined_metrics[[
        "variable", "depth_bin", "N", "model_mean", "observation_mean",
        "bias", "percent_bias", "MAE", "RMSD", "centred_RMSD",
        "correlation", "NSE",
    ]]
    combined_metrics.to_csv(
        new_output(review_dir / "tier1_all_variables_depth_metrics.csv"),
        index=False,
    )

    input_audit = pd.DataFrame(audit_rows)
    input_checksums = pd.DataFrame(checksum_rows).merge(
        input_audit, on="variable", validate="one_to_one"
    )
    input_checksums.to_csv(
        new_output(review_dir / "tier1_input_checksums.csv"), index=False
    )
else:
    combined_metrics = pd.read_csv(
        review_dir / "tier1_all_variables_depth_metrics.csv"
    )
    input_checksums = pd.read_csv(review_dir / "tier1_input_checksums.csv")

    audit_columns = [
        "variable", "input_rows", "finite_observations", "finite_model_values",
        "finite_pairs", "missing_or_nonfinite_observation",
        "missing_or_nonfinite_model", "missing_or_nonfinite_pair",
        "unbinned_finite_pairs",
    ]
    missing_audit = sorted(set(audit_columns).difference(input_checksums.columns))
    if missing_audit:
        raise ValueError(f"Input-checksum table is missing columns: {missing_audit}")
    input_audit = input_checksums[audit_columns].copy()

    for variable in variables:
        metrics = pd.read_csv(
            review_dir / variable / f"{variable}_{year}_depth_metrics.csv"
        )
        saved = pd.read_csv(
            review_dir / variable / f"{variable}_{year}_finite_pairs.csv"
        )

        required = {
            variable, "mod", "depth", "residual", "absolute_residual", "depth_bin"
        }
        missing = sorted(required.difference(saved.columns))
        if missing:
            raise ValueError(f"{variable}: saved finite-pair table is missing {missing}")

        tier1_tables[variable] = metrics
        tier1_data[variable] = saved
        tier1_pairs[variable] = pd.DataFrame({
            "observation": pd.to_numeric(saved[variable], errors="coerce"),
            "model": pd.to_numeric(saved["mod"], errors="coerce"),
            "depth": pd.to_numeric(saved["depth"], errors="coerce"),
            "residual": pd.to_numeric(saved["residual"], errors="coerce"),
            "absolute_residual": pd.to_numeric(
                saved["absolute_residual"], errors="coerce"
            ),
            "depth_bin": saved["depth_bin"],
        })

    print(f"Loaded metric tables: {list(tier1_tables)}")
    print(f"Loaded pair tables: {list(tier1_pairs)}")

input_audit

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharex=True)

for ax, variable in zip(axes.flat, variables):
    table = tier1_tables[variable].set_index("depth_bin").reindex(depth_labels)
    values = table["bias"].to_numpy(dtype=float)
    counts = table["N"].fillna(0).to_numpy(dtype=int)
    x = np.arange(len(depth_labels))
    bars = ax.bar(x, values, color="tab:blue")
    for bar, count in zip(bars, counts):
        bar.set_alpha(0.25 if count < 20 else 0.8)
    ax.axhline(0, color="black", linewidth=0.9)
    ax.set_title(variable)
    ax.set_ylabel(f"Bias ({variable_units[variable]})")
    ax.grid(axis="y", alpha=0.25)
    for xpos, value, count in zip(x, values, counts):
        if not np.isfinite(value):
            continue
        offset = 4 if value >= 0 else -12
        ax.annotate(
            f"N={count}", (xpos, value), xytext=(0, offset),
            textcoords="offset points", ha="center", fontsize=8,
            alpha=0.45 if count < 20 else 0.9,
        )
    ax.set_xticks(x, depth_labels, rotation=35, ha="right")

fig.suptitle("2014 Tier 1 depth-bin bias (model - observation)")
fig.tight_layout()
save_vertical_figure(fig, review_dir / "tier1_depth_bias_all_variables.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharex=True)

for ax, variable in zip(axes.flat, variables):
    table = tier1_tables[variable].set_index("depth_bin").reindex(depth_labels)
    values = table["RMSD"].to_numpy(dtype=float)
    counts = table["N"].fillna(0).to_numpy(dtype=int)
    x = np.arange(len(depth_labels))
    bars = ax.bar(x, values, color="tab:orange")
    for bar, count in zip(bars, counts):
        bar.set_alpha(0.25 if count < 20 else 0.8)
    ax.set_ylim(bottom=0)
    ax.set_title(variable)
    ax.set_ylabel(f"RMSD ({variable_units[variable]})")
    ax.grid(axis="y", alpha=0.25)
    for xpos, value, count in zip(x, values, counts):
        if not np.isfinite(value):
            continue
        ax.annotate(
            f"N={count}", (xpos, value), xytext=(0, 4),
            textcoords="offset points", ha="center", fontsize=8,
            alpha=0.45 if count < 20 else 0.9,
        )
    ax.set_xticks(x, depth_labels, rotation=35, ha="right")

fig.suptitle("2014 Tier 1 RMSD by observation-depth bin")
fig.tight_layout()
save_vertical_figure(fig, review_dir / "tier1_depth_rmsd_all_variables.png")
plt.show()

In [ ]:
print(f"Tier 1 mode: {review_mode}")
print(f"Variables ready: {list(tier1_tables)}")

In [ ]:
def percentile_limits(values, lower=1.0, upper=99.0, include_zero=False):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return (-1.0, 1.0)
    lo, hi = np.nanpercentile(values, [lower, upper])
    if include_zero:
        lo, hi = min(lo, 0.0), max(hi, 0.0)
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        pad = max(abs(lo) * 0.05, 1e-12)
        lo, hi = lo - pad, hi + pad
    else:
        pad = 0.04 * (hi - lo)
        lo, hi = lo - pad, hi + pad
    return lo, hi


for variable in variables:
    pairs = tier1_pairs[variable]
    overall = _metric_row(pairs)
    n = len(pairs)
    point_alpha = min(0.45, max(0.05, 1500.0 / max(n, 1)))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    pooled = np.concatenate([
        pairs["observation"].to_numpy(), pairs["model"].to_numpy()
    ])
    compare_limits = percentile_limits(pooled, 0.5, 99.5)
    axes[0].scatter(
        pairs["observation"], pairs["model"], s=9, alpha=point_alpha,
        edgecolors="none", rasterized=True,
    )
    axes[0].plot(compare_limits, compare_limits, "k--", linewidth=1)
    axes[0].set_xlim(compare_limits)
    axes[0].set_ylim(compare_limits)
    axes[0].set_xlabel(f"Observation ({variable_units[variable]})")
    axes[0].set_ylabel(f"Model ({variable_units[variable]})")
    axes[0].set_title("Observed versus model")

    residual_limit = (
        np.nanpercentile(np.abs(pairs["residual"]), 99.0) if n else 1.0
    )
    if not np.isfinite(residual_limit) or residual_limit == 0:
        residual_limit = max(
            float(np.nanmax(np.abs(pairs["residual"]))) if n else 1.0,
            1e-12,
        )
    axes[1].scatter(
        pairs["residual"], pairs["depth"], s=9, alpha=point_alpha,
        edgecolors="none", rasterized=True,
    )
    axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
    axes[1].set_xlim(-residual_limit, residual_limit)
    axes[1].set_xlabel(f"Residual ({variable_units[variable]})")
    axes[1].set_ylabel("Observation depth (m)")
    axes[1].set_title("Signed residual")
    axes[1].invert_yaxis()

    absolute_limits = percentile_limits(
        pairs["absolute_residual"], 0.0, 99.0, include_zero=True
    )
    axes[2].scatter(
        pairs["absolute_residual"], pairs["depth"], s=9,
        alpha=point_alpha, edgecolors="none", rasterized=True,
    )
    axes[2].set_xlim(absolute_limits)
    axes[2].set_xlabel(f"Absolute residual ({variable_units[variable]})")
    axes[2].set_ylabel("Observation depth (m)")
    axes[2].set_title("Absolute error")
    axes[2].invert_yaxis()

    for ax in axes:
        ax.grid(alpha=0.25)
    fig.suptitle(
        f"{variable}, {year}: N={n:,}, bias={overall['bias']:.4g}, "
        f"RMSD={overall['RMSD']:.4g}\n"
        "Axis limits use percentiles for display; CSV rows and metrics are untrimmed."
    )
    fig.tight_layout()
    save_vertical_figure(
        fig,
        review_dir / variable / f"{variable}_{year}_point_level_diagnostics.png",
    )
    if display_vertical_figures:
        plt.show()
    plt.close(fig)

In [ ]:
try:
    region_background = (
        xreg if 'xreg' in globals() else xr.load_dataset(diagnostic_regionfile)
    )
except (FileNotFoundError, OSError) as exc:
    print(f'Region file unavailable; skipping only geographic residual maps: {exc}')
else:
    for variable in variables:
        frame = tier1_data[variable]
        required = {variable, 'mod', 'lon', 'lat'}
        missing = sorted(required.difference(frame.columns))
        if missing:
            raise KeyError(f'{variable}: map cell requires columns {missing}')

        obs = pd.to_numeric(frame[variable], errors='coerce')
        mod_values = pd.to_numeric(frame['mod'], errors='coerce')
        lon = pd.to_numeric(frame['lon'], errors='coerce')
        lat = pd.to_numeric(frame['lat'], errors='coerce')
        map_mask = (
            np.isfinite(obs.to_numpy()) & np.isfinite(mod_values.to_numpy()) &
            np.isfinite(lon.to_numpy()) & np.isfinite(lat.to_numpy())
        )
        residual = (mod_values - obs).loc[map_mask]
        absolute_residual = residual.abs()
        map_lon = lon.loc[map_mask]
        map_lat = lat.loc[map_mask]

        signed_limit = np.nanpercentile(np.abs(residual), 98.0) if len(residual) else 1.0
        absolute_limit = np.nanpercentile(absolute_residual, 98.0) if len(residual) else 1.0
        signed_limit = signed_limit if np.isfinite(signed_limit) and signed_limit > 0 else 1.0
        absolute_limit = absolute_limit if np.isfinite(absolute_limit) and absolute_limit > 0 else 1.0

        fig, axes = plt.subplots(
            1, 2, figsize=(16, 7), subplot_kw={'projection': ccrs.PlateCarree()}
        )
        for ax in axes:
            region_background['region_id'].plot(
                ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False,
                cmap='Greys', alpha=0.12
            )
            ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.35)
            ax.coastlines(color='k')
            ax.add_feature(cf.BORDERS, color='darkgrey')
            ax.add_feature(cf.RIVERS, color='darkblue', alpha=0.5)

        signed = axes[0].scatter(
            map_lon, map_lat, c=residual, s=10, alpha=0.55,
            cmap='coolwarm', vmin=-signed_limit, vmax=signed_limit,
            transform=ccrs.PlateCarree(), edgecolors='none', rasterized=True
        )
        fig.colorbar(
            signed, ax=axes[0], extend='both', shrink=0.8,
            label=f'Residual ({variable_units[variable]})'
        )
        axes[0].set_title('Signed Residual (model - observation)')

        absolute = axes[1].scatter(
            map_lon, map_lat, c=absolute_residual, s=10, alpha=0.55,
            cmap='viridis', vmin=0, vmax=absolute_limit,
            transform=ccrs.PlateCarree(), edgecolors='none', rasterized=True
        )
        fig.colorbar(
            absolute, ax=axes[1], extend='max', shrink=0.8,
            label=f'Absolute residual ({variable_units[variable]})'
        )
        axes[1].set_title('Absolute error')
        fig.suptitle(
            f'{variable}, {year}: geographic residual diagnostics '
            '(colour limits use the 98th percentile)'
        )
        fig.tight_layout()
        save_vertical_figure(
            fig,
            review_dir / variable / f"{variable}_{year}_residual_maps.png",
        )
        if display_vertical_figures:
            plt.show()
        plt.close(fig)

In [ ]:
def value_and_bin_at_extreme(table, column, absolute=False):
    valid = table.loc[np.isfinite(table[column].to_numpy())]
    if valid.empty:
        return np.nan, None
    index = valid[column].abs().idxmax() if absolute else valid[column].idxmax()
    return valid.loc[index, column], valid.loc[index, 'depth_bin']


flag_rows = []
audit_by_variable = input_audit.set_index('variable')

for variable in variables:
    table = tier1_tables[variable]
    populated = table.loc[(table['N'] > 0) & np.isfinite(table['bias'])]
    populated_bias = populated['bias'].to_numpy(dtype=float)
    if populated_bias.size == 0:
        systematic_sign = 'no populated bins'
    elif np.all(populated_bias > 0):
        systematic_sign = 'positive'
    elif np.all(populated_bias < 0):
        systematic_sign = 'negative'
    elif np.all(populated_bias == 0):
        systematic_sign = 'zero'
    else:
        systematic_sign = 'mixed'

    largest_bias, largest_bias_bin = value_and_bin_at_extreme(
        table, 'bias', absolute=True
    )
    largest_rmsd, largest_rmsd_bin = value_and_bin_at_extreme(table, 'RMSD')
    overall = _metric_row(tier1_pairs[variable])
    audit = audit_by_variable.loc[variable]
    low_n_bins = table.loc[table['N'] < 20, 'depth_bin'].astype(str).tolist()
    missing_or_unbinned = bool(
        audit['missing_or_nonfinite_pair'] > 0 or audit['unbinned_finite_pairs'] > 0
    )

    flag_rows.append({
        'variable': variable,
        'finite_pairs': int(audit['finite_pairs']),
        'any_depth_bin_N_lt_20': bool(low_n_bins),
        'depth_bins_N_lt_20': ';'.join(low_n_bins),
        'largest_absolute_depth_bin_bias': abs(largest_bias) if np.isfinite(largest_bias) else np.nan,
        'signed_bias_at_largest_absolute_bin': largest_bias,
        'largest_absolute_bias_depth_bin': largest_bias_bin,
        'largest_depth_bin_RMSD': largest_rmsd,
        'largest_RMSD_depth_bin': largest_rmsd_bin,
        'systematic_bias_sign_across_populated_bins': systematic_sign,
        'overall_correlation': overall['correlation'],
        'correlation_below_0_5': bool(
            np.isfinite(overall['correlation']) and overall['correlation'] < 0.5
        ),
        'overall_NSE': overall['NSE'],
        'NSE_below_0': bool(np.isfinite(overall['NSE']) and overall['NSE'] < 0),
        'missing_or_nonfinite_pair_count': int(audit['missing_or_nonfinite_pair']),
        'unbinned_finite_pair_count': int(audit['unbinned_finite_pairs']),
        'missing_or_unbinned_data': missing_or_unbinned,
    })

review_flags = pd.DataFrame(flag_rows)
flags_output = managed_output_path(
    review_dir / "tier1_review_flags.csv",
    save_enabled=(review_mode == "create_new"),
    overwrite=overwrite_review_outputs,
)
if flags_output is not None:
    review_flags.to_csv(flags_output, index=False)
    print(f"Saved: {flags_output}")

review_order = ['temp', 'sal', 'po4', 'sio', 'nh4', 'chl', 'nox', 'oxy']
print(
    'Review order: temp/sal vertical behavior; po4/sio/nh4 depth structure; '
    'chl bloom structure; sparse nox; unit-provisional oxy.'
)
review_flags.set_index('variable').loc[review_order].reset_index()